# Nyagatare Yield Prediction — Data Preprocessing
**Capstone Project | Charlotte Kariza | ALU BSc Software Engineering**

This notebook:
1. Loads the combined RAB + Meteo Rwanda dataset
2. Separates the four record types
3. Cleans and encodes features
4. Matches climate data to crop growing seasons
5. Saves clean datasets ready for model training


## 0. Install & Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)
print('Libraries loaded.')

Libraries loaded.


## 1. Load the Combined Dataset

Upload your CSV file when prompted (Google Colab), or set the path if running locally.

In [2]:
# ── COLAB: upload file ──────────────────────────────────────────────────────
try:
    from google.colab import files
    uploaded = files.upload()  # Select your CSV here
    CSV_PATH = list(uploaded.keys())[0]
except ImportError:
    # Local path — change this to your file location
    CSV_PATH = '../data/raw/Combined_Nyagatare_Data_Master - Combined_Data.csv'

raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f'Loaded {len(raw)} rows × {len(raw.columns)} columns')
raw.head(3)

Loaded 491 rows × 65 columns


,Source File,Source Sheet,Record Type,Original Row,No,No 2,Code No,District,Sector,Cell,Village,Code,Longitude,Latitude,Altitude,Crop,Marshland,EC (µS/Cm),Total N (%),Field facilitator name,RAB-staff focal person,Season,Province,Farmer name,Variety,Trial type,"Slope trial located (top, middle, valley)",Previous crop,Planting date,Treatment,Germination date,Number of plants germinated /plot or treatment,Flowering date (at 50%) per plot/Treatment,Number of plants harvested/ net plot,Number of pods per plot/treatment (5plants),Harvesting date,Harvesting Net Plot (m2),Fresh Grain Weight (kg),Grain Yield (t/ha),Marshland name,Date of basal fertilizer application,At 25- 30 days after Transplanting,At 45- 50 days after Transplanting,At 75- 80 days after Transplanting,"Crop management score; 1=OK, 2=Average, 3=Not OK","Trial damage score; 1=Damage, 2=Moderate damage, 3=No damage",plant height (10 plans per plot)_cm,No. of tillers/10 plants/plot,Length of panicles/plot(cm),Field Fresh Grain Weight (kg/net plot),Paddy Moisture at Harvest (%),Paddy Yield (t/ha at 18% MC),fresh weight of biomass kg/net plot,Weight of fresh biomass sub-sample kg/ net plot,Weight of dry biomass sub-sample (kg),dry weight of biomass kg/net plot,Station Name,Station Latitude,Station Longitude,Station Elevation,Year,Month,Maximum Temperature,Relative Humidity,Rainfall (mm)
0,Muvumba_Nyagatare_Tot_N.xlsx,Sheet1,Soil/Nutrient,3,1.0,1.0,C1,Nyagatare,Tabagwe,Gitengure,Nshuri,NYATAGINSHU8,195875.0,9855159.0,1345.0,Rice,MUVUMBA,267.0,0.21182,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Muvumba_Nyagatare_Tot_N.xlsx,Sheet1,Soil/Nutrient,4,2.0,2.0,C2,Nyagatare,Tabagwe,Gitengure,Nshuri,NYATAGINSHU7,199460.0,9855343.0,1348.0,Rice,MUVUMBA,131.0,0.32466,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Muvumba_Nyagatare_Tot_N.xlsx,Sheet1,Soil/Nutrient,5,3.0,3.0,C3,Nyagatare,Tabagwe,Gitengure,Nshuri,NYATAGINSHU6,200163.0,9854942.0,1345.0,Rice,MUVUMBA,219.0,0.27958,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Identify & Separate Record Types

The combined CSV has 4 record types mixed together:
- **Soil/Nutrient records** — have EC / Total N columns, no yield
- **Bean crop trials** — `Crop` column contains 'bean', target: `Grain Yield (t/ha)`
- **Rice crop trials** — `Crop` column contains 'rice', target: `Paddy Yield (t/ha at 18% MC)`
- **Climate records** — have `Station Name` / `Rainfall (mm)` columns, no crop info

In [3]:
# Normalise column names: strip whitespace
raw.columns = raw.columns.str.strip()

# ── Detect climate rows by presence of 'Rainfall (mm)' or 'Station Name' ───
climate_col_candidates = [c for c in raw.columns if 'Rainfall' in c or 'Station' in c]
print('Climate-indicator columns found:', climate_col_candidates)

# ── Use the 'Crop' column to split bean vs rice ──────────────────────────────
if 'Crop' in raw.columns:
    crop_col = 'Crop'
else:
    # Find the column that holds 'Bean' or 'Rice' values
    crop_col = next(
        (c for c in raw.columns
         if raw[c].dropna().astype(str).str.lower().str.contains('bean|rice').any()),
        None
    )
    print(f'Crop column detected as: {crop_col}')

# Normalise Crop values to lowercase for matching
if crop_col:
    raw['_crop_lower'] = raw[crop_col].astype(str).str.strip().str.lower()

# ── Climate rows: Station Name is filled ────────────────────────────────────
if 'Station Name' in raw.columns:
    climate_mask = raw['Station Name'].notna() & (raw['Station Name'].astype(str).str.strip() != '')
elif climate_col_candidates:
    climate_mask = raw[climate_col_candidates[0]].notna()
else:
    climate_mask = pd.Series([False] * len(raw))

df_climate = raw[climate_mask].copy()
df_crop    = raw[~climate_mask].copy()

# ── Bean vs Rice vs Soil ─────────────────────────────────────────────────────
df_beans = df_crop[df_crop['_crop_lower'].str.contains('bean', na=False)].copy()
df_rice  = df_crop[df_crop['_crop_lower'].str.contains('rice', na=False)].copy()

# Soil/Nutrient rows: crop column is empty AND no yield data
yield_cols = [c for c in df_crop.columns if 'yield' in c.lower() or 'Yield' in c]
df_soil = df_crop[
    ~df_crop['_crop_lower'].str.contains('bean|rice', na=False)
].copy()

print(f'\nRecord counts after split:')
print(f'  Climate rows : {len(df_climate)}')
print(f'  Bean trials  : {len(df_beans)}')
print(f'  Rice trials  : {len(df_rice)}')
print(f'  Soil/other   : {len(df_soil)}')

Climate-indicator columns found: ['Station Name', 'Station Latitude', 'Station Longitude', 'Station Elevation', 'Rainfall (mm)']

Record counts after split:
  Climate rows : 228
  Bean trials  : 96
  Rice trials  : 167
  Soil/other   : 0


## 3. Clean the Climate Data

In [4]:
# Keep only Nyagatare station records
if 'Station Name' in df_climate.columns:
    df_climate = df_climate[
        df_climate['Station Name'].astype(str).str.upper().str.contains('NYAGATARE')
    ].copy()

def find_col(df, *keywords):
    for kw in keywords:
        for c in df.columns:
            if kw.lower() in c.lower():
                return c
    return None

year_col   = find_col(df_climate, 'Year')
month_col  = find_col(df_climate, 'Month')
rain_col   = find_col(df_climate, 'Rainfall', 'Rain')
temp_col   = find_col(df_climate, 'Temperature', 'Temp')
humid_col  = find_col(df_climate, 'Humidity', 'Humid')

print(f'Year={year_col} | Month={month_col} | Rain={rain_col} | Temp={temp_col} | Humid={humid_col}')

# Extract relevant columns and rename
climate_cols = {c: c for c in [year_col, month_col, rain_col, temp_col, humid_col] if c}
df_clim_raw = df_climate[list(climate_cols.keys())].copy()
df_clim_raw.rename(columns={
    year_col:  'Year',
    month_col: 'Month',
    rain_col:  'Rainfall_mm',
    temp_col:  'Max_Temp_C',
    humid_col: 'Humidity_pct'
}, inplace=True)

for col in ['Year', 'Month', 'Rainfall_mm', 'Max_Temp_C', 'Humidity_pct']:
    if col in df_clim_raw.columns:
        df_clim_raw[col] = pd.to_numeric(df_clim_raw[col], errors='coerce')

df_clim_raw.dropna(subset=['Year', 'Month'], inplace=True)

# Temperature/Humidity and Rainfall come from separate source rows in the CSV.
# Merge them into one row per Year+Month by taking the first non-null value.
df_clim_clean = (
    df_clim_raw
    .groupby(['Year', 'Month'], as_index=False)
    .first()
    .sort_values(['Year', 'Month'])
    .reset_index(drop=True)
)

print(f'\nClimate records after merging temp+rainfall rows: {len(df_clim_clean)}')
print(f'Missing values:\n{df_clim_clean.isnull().sum()}')
df_clim_clean.head(8)

Year=Year | Month=Month | Rain=Rainfall (mm) | Temp=Maximum Temperature | Humid=Relative Humidity

Climate records after merging temp+rainfall rows: 131
Missing values:
Year             0
Month            0
Rainfall_mm     33
Max_Temp_C       4
Humidity_pct    37
dtype: int64


,Year,Month,Rainfall_mm,Max_Temp_C,Humidity_pct
0,2015.0,1.0,0.2,28.25,NaN
1,2015.0,2.0,45.0,28.93,NaN
2,2015.0,3.0,52.2,28.73,NaN
3,2015.0,4.0,105.8,16.71,NaN
4,2015.0,5.0,NaN,0.00,NaN
5,2015.0,6.0,NaN,0.00,NaN
6,2015.0,7.0,NaN,0.00,NaN
7,2015.0,8.0,NaN,10.63,NaN


## 4. Encode the Treatment Column (NPK levels)

The Treatment column encodes fertiliser levels like `NPK 10N`, `NPK 30P`, `Control`, etc.
We extract three numeric features: `N_level`, `P_level`, `K_level` (kg/ha equivalents coded 0–3).

In [5]:
def encode_treatment(treatment_str):
    """
    Returns (has_N, has_P, has_K, N_boost, P_boost, K_boost).
    *_boost captures extra application levels coded 0–3.
    Works for both bean treatments (e.g. 'NPK (30N)') and
    rice treatments (e.g. 'NPK (60N)', 'NPK 17*3').
    """
    s = str(treatment_str).strip().upper()

    if 'CONTROL' in s:
        return 0, 0, 0, 0, 0, 0

    if s == 'NP':
        return 1, 1, 0, 0, 0, 0
    if s == 'NK':
        return 1, 0, 1, 0, 0, 0
    if s == 'PK':
        return 0, 1, 1, 0, 0, 0

    has_N = 1 if 'N' in s else 0
    has_P = 1 if 'P' in s else 0
    has_K = 1 if 'K' in s else 0

    # Boost level: digit directly before the nutrient letter (handles both
    # 'NPK (60N)' and 'NPK 60N' formats due to regex ignoring parentheses)
    def boost(letter):
        match = re.search(r'(\d+)\s*' + letter, s)
        if match:
            val = int(match.group(1))
            if val <= 10:   return 1
            elif val <= 30: return 2
            else:           return 3
        return 0

    n_boost = boost('N')
    p_boost = boost('P')
    k_boost = boost('K')

    # Named high-level treatments
    if 'INCREASED NPK2' in s:
        return 1, 1, 1, 3, 3, 3
    if 'INCREASED NPK1' in s:
        return 1, 1, 1, 2, 2, 2
    if 'INCREASED NPK' in s:   # rice uses this without a number
        return 1, 1, 1, 2, 2, 2

    return has_N, has_P, has_K, n_boost, p_boost, k_boost


# Verify encoding on all treatment types present in the data
test_treatments = [
    'Control', 'NPK', 'NP', 'NK', 'PK',
    'NPK (10N)', 'NPK (30N)', 'NPK (10P)', 'NPK (30P)', 'NPK (40P)',
    'NPK (10K)', 'NPK (30K)', 'NPK (40K)',
    'NPK (60N)', 'NPK (100N)', 'NPK (120N)',   # rice N levels
    'NPK (45P)', 'NPK (60P)',                   # rice P levels
    'NPK (20K)', 'NPK (40K)',                   # rice K levels
    'NPK 17*3', 'NPK 17*3_later K',            # rice baseline treatments
    'Increased NPK1', 'Increased NPK2', 'Increased NPK',
]

test_df = pd.DataFrame(
    [encode_treatment(t) for t in test_treatments],
    columns=['has_N', 'has_P', 'has_K', 'N_boost', 'P_boost', 'K_boost'],
    index=test_treatments
)
print(test_df)

                  has_N  has_P  has_K  N_boost  P_boost  K_boost
Control               0      0      0        0        0        0
NPK                   1      1      1        0        0        0
NP                    1      1      0        0        0        0
NK                    1      0      1        0        0        0
PK                    0      1      1        0        0        0
NPK (10N)             1      1      1        1        0        0
NPK (30N)             1      1      1        2        0        0
NPK (10P)             1      1      1        0        1        0
NPK (30P)             1      1      1        0        2        0
NPK (40P)             1      1      1        0        3        0
NPK (10K)             1      1      1        0        0        1
NPK (30K)             1      1      1        0        0        2
NPK (40K)             1      1      1        0        0        3
NPK (60N)             1      1      1        3        0        0
NPK (100N)            1  

## 5. Clean the Bean Crop Trials

In [6]:
# ── Identify key columns by keyword search ───────────────────────────────────
def find_col_in(df, *keywords):
    for kw in keywords:
        for c in df.columns:
            if kw.lower() in c.lower():
                return c
    return None

# Bean target column
bean_yield_col = find_col_in(df_beans, 'Grain Yield', 'grain yield')
treatment_col  = find_col_in(df_beans, 'Treatment')
variety_col    = find_col_in(df_beans, 'Variety')
slope_col      = find_col_in(df_beans, 'Slope')
prev_crop_col  = find_col_in(df_beans, 'Previous crop', 'Previous Crop')
season_col     = find_col_in(df_beans, 'Season')
sector_col     = find_col_in(df_beans, 'Sector')
plant_date_col = find_col_in(df_beans, 'Planting date', 'Planting Date')
harv_date_col  = find_col_in(df_beans, 'Harvesting date', 'Harvest')
plot_size_col  = find_col_in(df_beans, 'Net plot', 'plot size')
germ_col       = find_col_in(df_beans, 'Plants germinated', 'Germinated')
harv_pl_col    = find_col_in(df_beans, 'Plants harvested', 'harvested')

print('Bean column mapping:')
for name, col in [
    ('Yield target', bean_yield_col), ('Treatment', treatment_col),
    ('Variety', variety_col), ('Slope', slope_col),
    ('Previous crop', prev_crop_col), ('Season', season_col),
    ('Sector', sector_col), ('Planting date', plant_date_col),
    ('Harvesting date', harv_date_col), ('Plot size', plot_size_col)
]:
    print(f'  {name:20s} → {col}')

Bean column mapping:
  Yield target         → Grain Yield (t/ha)
  Treatment            → Treatment
  Variety              → Variety
  Slope                → Slope trial located (top, middle, valley)
  Previous crop        → Previous crop
  Season               → Season
  Sector               → Sector
  Planting date        → Planting date
  Harvesting date      → Harvesting date
  Plot size            → Number of plants harvested/ net plot


In [7]:
# ── Apply NPK encoding ────────────────────────────────────────────────────────
npk_encoded = df_beans[treatment_col].apply(encode_treatment)
df_beans[['has_N','has_P','has_K','N_boost','P_boost','K_boost']] = pd.DataFrame(
    npk_encoded.tolist(), index=df_beans.index
)

# ── Parse planting and harvest dates ─────────────────────────────────────────
df_beans['planting_date']   = pd.to_datetime(df_beans[plant_date_col], dayfirst=True, errors='coerce')
df_beans['harvesting_date'] = pd.to_datetime(df_beans[harv_date_col],  dayfirst=True, errors='coerce')

df_beans['planting_month'] = df_beans['planting_date'].dt.month
df_beans['planting_year']  = df_beans['planting_date'].dt.year
df_beans['harvest_month']  = df_beans['harvesting_date'].dt.month
df_beans['harvest_year']   = df_beans['harvesting_date'].dt.year
df_beans['growing_days']   = (df_beans['harvesting_date'] - df_beans['planting_date']).dt.days

# ── Convert yield to numeric ──────────────────────────────────────────────────
df_beans['yield_t_ha'] = pd.to_numeric(df_beans[bean_yield_col], errors='coerce')

# ── Drop rows with missing yield (cannot train on them) ───────────────────────
before = len(df_beans)
df_beans.dropna(subset=['yield_t_ha'], inplace=True)
print(f'Bean rows after dropping missing yield: {len(df_beans)} (dropped {before - len(df_beans)})')

# ── Encode slope position ─────────────────────────────────────────────────────
slope_map = {'flat': 0, 'gentle': 1, 'moderate': 2, 'steep': 3}
df_beans['slope_encoded'] = (
    df_beans[slope_col].astype(str).str.strip().str.lower()
    .map(slope_map)
    .fillna(0)
    .astype(int)
)

# ── Label encode Variety, Previous crop, Sector ───────────────────────────────
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col_name, new_col in [
    (variety_col,   'variety_encoded'),
    (prev_crop_col, 'prev_crop_encoded'),
    (sector_col,    'sector_encoded'),
]:
    if col_name and col_name in df_beans.columns:
        df_beans[new_col] = le.fit_transform(
            df_beans[col_name].astype(str).str.strip().str.lower()
        )

print(df_beans[['yield_t_ha', 'has_N','has_P','has_K','N_boost','slope_encoded',
                'variety_encoded','planting_year','planting_month','growing_days']].describe())

Bean rows after dropping missing yield: 96 (dropped 0)


/tmp/ipykernel_72474/412674573.py:9: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_beans['harvesting_date'] = pd.to_datetime(df_beans[harv_date_col],  dayfirst=True, errors='coerce')


       yield_t_ha      has_N      has_P      has_K    N_boost  slope_encoded  \
count   96.000000  96.000000  96.000000  96.000000  96.000000           96.0   
mean     2.497500   0.875000   0.875000   0.812500   0.500000            0.0   
std      0.447159   0.332455   0.332455   0.392361   0.940325            0.0   
min      1.000000   0.000000   0.000000   0.000000   0.000000            0.0   
25%      2.300000   1.000000   1.000000   1.000000   0.000000            0.0   
50%      2.510000   1.000000   1.000000   1.000000   0.000000            0.0   
75%      2.820000   1.000000   1.000000   1.000000   0.250000            0.0   
max      3.250000   1.000000   1.000000   1.000000   3.000000            0.0   

       variety_encoded  planting_year  planting_month  growing_days  
count             96.0           80.0       80.000000      80.00000  
mean               0.0         2021.0        9.000000      97.00000  
std                0.0            0.0        0.636446      18.50077  

## 6. Clean the Rice Crop Trials

In [8]:
# Rice target: Paddy Yield (t/ha at 18% MC)
rice_yield_col  = find_col_in(df_rice, 'Paddy Yield', 'paddy yield')
treatment_col_r = find_col_in(df_rice, 'Treatment')
variety_col_r   = find_col_in(df_rice, 'Variety')
slope_col_r     = find_col_in(df_rice, 'Slope')
prev_crop_col_r = find_col_in(df_rice, 'Previous crop', 'Previous Crop')
sector_col_r    = find_col_in(df_rice, 'Sector')
plant_date_r    = find_col_in(df_rice, 'Planting date', 'Planting Date')
harv_date_r     = find_col_in(df_rice, 'Harvesting date', 'Harvest')

print('Rice column mapping:')
for name, col in [
    ('Yield target', rice_yield_col), ('Treatment', treatment_col_r),
    ('Planting date', plant_date_r),  ('Harvesting date', harv_date_r),
    ('Sector', sector_col_r),
]:
    print(f'  {name:20s} → {col}')

# NPK
npk_r = df_rice[treatment_col_r].apply(encode_treatment)
df_rice[['has_N','has_P','has_K','N_boost','P_boost','K_boost']] = pd.DataFrame(
    npk_r.tolist(), index=df_rice.index
)

# Dates — rice harvest date is only filled for the FIRST treatment row of
# each farmer's plot block. Forward-fill so all rows in the block get it.
df_rice[harv_date_r] = df_rice[harv_date_r].replace('', np.nan)
df_rice[harv_date_r] = df_rice[harv_date_r].ffill()

df_rice['planting_date']   = pd.to_datetime(df_rice[plant_date_r], dayfirst=True, errors='coerce')
df_rice['harvesting_date'] = pd.to_datetime(df_rice[harv_date_r],  dayfirst=True, errors='coerce')
df_rice['planting_month']  = df_rice['planting_date'].dt.month
df_rice['planting_year']   = df_rice['planting_date'].dt.year
df_rice['harvest_month']   = df_rice['harvesting_date'].dt.month
df_rice['harvest_year']    = df_rice['harvesting_date'].dt.year
df_rice['growing_days']    = (df_rice['harvesting_date'] - df_rice['planting_date']).dt.days

# Fix data-entry error: 12 rows have harvest year written as 2021 instead of 2022,
# producing negative growing_days (-55). Replace with the median of valid rows
# from the same planting month (Feb = ~132 days).
neg_mask = df_rice['growing_days'] < 0
if neg_mask.any():
    for month in df_rice.loc[neg_mask, 'planting_month'].unique():
        valid_days = df_rice.loc[(df_rice['planting_month'] == month) & (~neg_mask), 'growing_days']
        median_val = valid_days.median()
        df_rice.loc[neg_mask & (df_rice['planting_month'] == month), 'growing_days'] = median_val
    print(f'Fixed {neg_mask.sum()} rows with negative growing_days → median imputed')

# Yield
df_rice['yield_t_ha'] = pd.to_numeric(df_rice[rice_yield_col], errors='coerce')
before = len(df_rice)
df_rice.dropna(subset=['yield_t_ha'], inplace=True)
print(f'\nRice rows after dropping missing yield: {len(df_rice)} (dropped {before - len(df_rice)})')
print(f'Missing harvest dates after ffill: {df_rice["harvesting_date"].isna().sum()}')

# Slope + categoricals
df_rice['slope_encoded'] = (
    df_rice[slope_col_r].astype(str).str.strip().str.lower()
    .map(slope_map).fillna(0).astype(int)
)
for col_name, new_col in [
    (variety_col_r,   'variety_encoded'),
    (prev_crop_col_r, 'prev_crop_encoded'),
    (sector_col_r,    'sector_encoded'),
]:
    if col_name and col_name in df_rice.columns:
        df_rice[new_col] = le.fit_transform(
            df_rice[col_name].astype(str).str.strip().str.lower()
        )

print(df_rice[['yield_t_ha','has_N','has_P','has_K','slope_encoded','growing_days']].describe())

Rice column mapping:
  Yield target         → Paddy Yield (t/ha at 18% MC)
  Treatment            → Treatment
  Planting date        → Planting date
  Harvesting date      → Harvesting date
  Sector               → Sector
Fixed 12 rows with negative growing_days → median imputed

Rice rows after dropping missing yield: 120 (dropped 47)
Missing harvest dates after ffill: 0
       yield_t_ha       has_N       has_P       has_K  slope_encoded  \
count  120.000000  120.000000  120.000000  120.000000          120.0   
mean     6.366579    0.916667    0.916667    0.916667            0.0   
std      1.491689    0.277544    0.277544    0.277544            0.0   
min      2.298109    0.000000    0.000000    0.000000            0.0   
25%      5.813053    1.000000    1.000000    1.000000            0.0   
50%      6.520566    1.000000    1.000000    1.000000            0.0   
75%      7.194101    1.000000    1.000000    1.000000            0.0   
max      9.117305    1.000000    1.000000    1.00

/tmp/ipykernel_72474/1376937711.py:30: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_rice['planting_date']   = pd.to_datetime(df_rice[plant_date_r], dayfirst=True, errors='coerce')
/tmp/ipykernel_72474/1376937711.py:31: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_rice['harvesting_date'] = pd.to_datetime(df_rice[harv_date_r],  dayfirst=True, errors='coerce')


## 7. Match Climate Data to Crop Growing Seasons

For each trial, we compute:
- **total_season_rainfall_mm** — sum of monthly rainfall during the growing months
- **mean_season_temp_C** — mean of monthly max temperature during the growing months

In [9]:
def get_season_climate(planting_year, planting_month, harvest_year, harvest_month, df_clim):
    """
    Sum rainfall and average temperature for all months
    between planting and harvest (inclusive).
    Returns (total_rainfall_mm, mean_temp_C) or (NaN, NaN) if data missing.
    """
    if pd.isna(planting_year) or pd.isna(planting_month):
        return np.nan, np.nan

    # Build list of (year, month) tuples in the growing period
    periods = []
    y, m = int(planting_year), int(planting_month)
    hy, hm = int(harvest_year), int(harvest_month)
    while (y, m) <= (hy, hm):
        periods.append((y, m))
        m += 1
        if m > 12:
            m = 1
            y += 1
        if len(periods) > 18:  # safety cap
            break

    mask = df_clim.apply(lambda r: (r['Year'], r['Month']) in periods, axis=1)
    subset = df_clim[mask]

    if subset.empty:
        return np.nan, np.nan

    total_rain = subset['Rainfall_mm'].sum() if 'Rainfall_mm' in subset.columns else np.nan
    mean_temp  = subset['Max_Temp_C'].mean()  if 'Max_Temp_C'  in subset.columns else np.nan
    return total_rain, mean_temp


# ── Apply to beans ────────────────────────────────────────────────────────────
print('Matching climate to bean trials...')
bean_climate = df_beans.apply(
    lambda r: pd.Series(
        get_season_climate(
            r['planting_year'], r['planting_month'],
            r['harvest_year'],  r['harvest_month'],
            df_clim_clean
        ),
        index=['total_rainfall_mm', 'mean_temp_C']
    ), axis=1
)
df_beans[['total_rainfall_mm', 'mean_temp_C']] = bean_climate

# ── Apply to rice ─────────────────────────────────────────────────────────────
print('Matching climate to rice trials...')
rice_climate = df_rice.apply(
    lambda r: pd.Series(
        get_season_climate(
            r['planting_year'], r['planting_month'],
            r['harvest_year'],  r['harvest_month'],
            df_clim_clean
        ),
        index=['total_rainfall_mm', 'mean_temp_C']
    ), axis=1
)
df_rice[['total_rainfall_mm', 'mean_temp_C']] = rice_climate

print('\nBean climate match coverage:')
print(df_beans[['total_rainfall_mm','mean_temp_C']].notna().sum())
print('\nRice climate match coverage:')
print(df_rice[['total_rainfall_mm','mean_temp_C']].notna().sum())

Matching climate to bean trials...


Matching climate to rice trials...



Bean climate match coverage:
total_rainfall_mm    80
mean_temp_C          80
dtype: int64

Rice climate match coverage:
total_rainfall_mm    108
mean_temp_C          108
dtype: int64


## 8. Build Final Model-Ready Datasets

In [10]:
FEATURE_COLS = [
    'has_N', 'has_P', 'has_K', 'N_boost', 'P_boost', 'K_boost',
    'slope_encoded', 'variety_encoded', 'prev_crop_encoded', 'sector_encoded',
    'planting_month', 'growing_days',
    'total_rainfall_mm', 'mean_temp_C'
]
TARGET_COL = 'yield_t_ha'

def make_model_df(df, label):
    cols = [c for c in FEATURE_COLS if c in df.columns] + [TARGET_COL]
    out = df[cols].copy()
    out['crop'] = label
    # Fill remaining NaNs with column median (for climate gaps)
    for c in out.select_dtypes(include='number').columns:
        out[c] = out[c].fillna(out[c].median())
    print(f'{label}: {len(out)} rows, {out[TARGET_COL].isna().sum()} missing yield')
    return out

df_beans_model = make_model_df(df_beans, 'bean')
df_rice_model  = make_model_df(df_rice,  'rice')

print('\n── Bean model dataset ──')
print(df_beans_model.describe().round(2))
print('\n── Rice model dataset ──')
print(df_rice_model.describe().round(2))

bean: 96 rows, 0 missing yield
rice: 120 rows, 0 missing yield

── Bean model dataset ──
       has_N  has_P  has_K  N_boost  P_boost  K_boost  slope_encoded  \
count  96.00  96.00  96.00    96.00    96.00    96.00           96.0   
mean    0.88   0.88   0.81     0.50     0.69     0.69            0.0   
std     0.33   0.33   0.39     0.94     1.11     1.11            0.0   
min     0.00   0.00   0.00     0.00     0.00     0.00            0.0   
25%     1.00   1.00   1.00     0.00     0.00     0.00            0.0   
50%     1.00   1.00   1.00     0.00     0.00     0.00            0.0   
75%     1.00   1.00   1.00     0.25     1.25     1.25            0.0   
max     1.00   1.00   1.00     3.00     3.00     3.00            0.0   

       variety_encoded  prev_crop_encoded  sector_encoded  planting_month  \
count             96.0              96.00            96.0           96.00   
mean               0.0               0.50             0.5            9.00   
std                0.0         

## 9. Save Clean Datasets

In [11]:
import os

# ── Local save ────────────────────────────────────────────────────────────────
os.makedirs('../data/processed', exist_ok=True)
df_beans_model.to_csv('../data/processed/beans_clean.csv', index=False)
df_rice_model.to_csv('../data/processed/rice_clean.csv',   index=False)
df_clim_clean.to_csv('../data/processed/climate_clean.csv', index=False)

print('Saved:')
print('  ../data/processed/beans_clean.csv')
print('  ../data/processed/rice_clean.csv')
print('  ../data/processed/climate_clean.csv')

# ── Colab: also download to your computer ────────────────────────────────────
try:
    from google.colab import files
    files.download('../data/processed/beans_clean.csv')
    files.download('../data/processed/rice_clean.csv')
    print('Files downloaded to your computer.')
except ImportError:
    pass  # Not in Colab — files saved locally above

Saved:
  ../data/processed/beans_clean.csv
  ../data/processed/rice_clean.csv
  ../data/processed/climate_clean.csv


---
**Next step:** Open `02_model_training.ipynb` to train Random Forest and Gradient Boosting models on the cleaned data.